<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div><strong>Universidad Externado de Colombia</strong><br>
  <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
  <span>Docente: Wilmer Pineda-Ríos</span></div>
  <img src="../../assets/brand/logo-externado.png" width="190">
</div>

# Sesión 2 — Complejidad, validación y poda

**Objetivo.** Seleccionar la complejidad del árbol sin utilizar el conjunto de prueba como herramienta de ajuste.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
candidates = [
    Path("../../datasets/public/Bike_Sharing_Day.csv"),
    Path("datasets/public/Bike_Sharing_Day.csv"),
    Path("../datasets/public/Bike_Sharing_Day.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("No se encontró Bike_Sharing_Day.csv")

df = pd.read_csv(data_path, parse_dates=["dteday"]).sort_values("dteday").reset_index(drop=True)
df.head()

In [ ]:
target = "cnt"
leakage = ["casual", "registered"]
drop_columns = ["instant", "dteday", target, *leakage]
X = df.drop(columns=drop_columns)
y = df[target]

categorical = ["season", "mnth", "weekday", "weathersit"]
numeric = [c for c in X.columns if c not in categorical]

cut = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

preprocess = ColumnTransformer(
    [("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    remainder="passthrough",
)
cv = TimeSeriesSplit(n_splits=5)
X_train.shape, X_test.shape, (df.loc[cut, "dteday"], df["dteday"].max())

In [ ]:
def metrics(name, y_true, y_pred):
    return {
        "modelo": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

## 1. Complejidad como decisión

Un árbol profundo reduce el error de entrenamiento, pero puede aprender particularidades inestables. Observaremos simultáneamente entrenamiento y validación.

In [ ]:
depths = range(1, 16)
rows = []
for depth in depths:
    model = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(max_depth=depth, min_samples_leaf=5, random_state=RANDOM_STATE))])
    score = cross_validate(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error", return_train_score=True)
    rows.append({"max_depth": depth, "MAE_train": -score["train_score"].mean(), "MAE_valid": -score["test_score"].mean(), "SD_valid": score["test_score"].std()})
depth_results = pd.DataFrame(rows)
depth_results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depth_results["max_depth"], depth_results["MAE_train"], marker="o", label="entrenamiento")
ax.plot(depth_results["max_depth"], depth_results["MAE_valid"], marker="o", label="validación")
ax.set(xlabel="max_depth", ylabel="MAE", title="La mejor complejidad no se decide con entrenamiento")
ax.legend()
plt.show()

## 2. Tamaño mínimo de hoja

In [ ]:
leaf_sizes = [1, 3, 5, 10, 20, 35, 50]
rows = []
for leaf in leaf_sizes:
    model = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(max_depth=None, min_samples_leaf=leaf, random_state=RANDOM_STATE))])
    score = cross_validate(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    rows.append({"min_samples_leaf": leaf, "MAE_valid": -score["test_score"].mean(), "SD": score["test_score"].std()})
pd.DataFrame(rows).round(2)

## 3. Poda costo-complejidad

La poda minimiza $R_\alpha(T)=R(T)+\alpha|T_{hojas}|$. Primero obtenemos la ruta de subárboles y luego elegimos $\alpha$ con validación.

In [ ]:
prep_fitted = preprocess.fit(X_train)
X_train_t = prep_fitted.transform(X_train)
base_tree = DecisionTreeRegressor(random_state=RANDOM_STATE, min_samples_leaf=3)
path = base_tree.cost_complexity_pruning_path(X_train_t, y_train)
alphas = np.unique(np.quantile(path.ccp_alphas[:-1], np.linspace(0, 1, 16)))
alphas

In [ ]:
pruning_rows = []
for alpha in alphas:
    model = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(ccp_alpha=float(alpha), min_samples_leaf=3, random_state=RANDOM_STATE))])
    score = cross_validate(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    pruning_rows.append({"ccp_alpha": alpha, "MAE_valid": -score["test_score"].mean(), "SD": score["test_score"].std()})
pruning = pd.DataFrame(pruning_rows)
pruning.sort_values("MAE_valid").head().round(3)

In [ ]:
best_alpha = float(pruning.loc[pruning["MAE_valid"].idxmin(), "ccp_alpha"])
final_model = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(ccp_alpha=best_alpha, min_samples_leaf=3, random_state=RANDOM_STATE))])
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)
pd.DataFrame([metrics("árbol podado", y_test, test_pred)]).round(2)

## 4. Test se consulta una sola vez

El valor anterior es una estimación final. Cambiar ahora `ccp_alpha` para mejorar test convertiría test en parte de la selección.

In [ ]:
residuals = y_test.to_numpy() - test_pred
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(test_pred, residuals, alpha=.65, color="#0f766e")
ax.axhline(0, color="#172029", linewidth=1)
ax.set(xlabel="predicción", ylabel="real - predicción", title="Residuales del árbol seleccionado")
plt.show()

## 5. Salida

1. ¿Qué evidencia distingue underfitting de overfitting?
2. ¿Qué regularizan `max_depth`, `min_samples_leaf` y `ccp_alpha`?
3. ¿Dónde debe vivir el preprocesamiento durante CV?
4. ¿Por qué no volvemos a ajustar después de mirar test?